In [ ]:
import sqlite3, os
import pandas as pd

DATA = "."
DB = "telecom_rt.db"
con = sqlite3.connect(DB)
con.executescript("""
CREATE TABLE IF NOT EXISTS kpi_readings (
reading_ts TEXT NOT NULL,
cell_id TEXT NOT NULL,
region TEXT NOT NULL,
latency_ms REAL,
throughput_mbps REAL,
downtime_min INTEGER NOT NULL DEFAULT 0,
drop_call_rate_pct REAL,
UNIQUE (cell_id, reading_ts)
);
CREATE INDEX IF NOT EXISTS ix_kpi_cell_ts ON kpi_readings (cell_id, reading_ts);
""")
con.commit()

In [ ]:
raw = pd.read_csv(f"{DATA}/kpi_stream_seed.csv")
print("rows in file             :", len(raw))
print("distinct cells           :", raw.cell_id.nunique())
print(
    "duplicate (cell_id, ts)  :", raw.duplicated(subset=["cell_id", "reading_ts"]).sum()
)
print("blank latency_ms         :", raw.latency_ms.isna().sum())
print("negative throughput_mbps :", (raw.throughput_mbps < 0).sum())
print("rows with downtime > 0   :", (raw.downtime_min > 0).sum())

In [ ]:
mw = pd.read_csv(f"{DATA}/maintenance_windows.csv")
raw["reading_ts"] = pd.to_datetime(raw.reading_ts)
mw["window_start"] = pd.to_datetime(mw.window_start)
mw["window_end"] = pd.to_datetime(mw.window_end)

in_maint = pd.Series(False, index=raw.index)
for _, w in mw.iterrows():
    in_maint |= (
        (raw.cell_id == w.cell_id)
        & (raw.reading_ts >= w.window_start)
        & (raw.reading_ts < w.window_end)
    )

blank = raw.latency_ms.isna()
print("blank inside a maintenance window :", int((blank & in_maint).sum()))
print("blank outside any window          :", int((blank & ~in_maint).sum()))

In [ ]:
clean = raw.drop_duplicates(subset=["cell_id", "reading_ts"], keep="first").copy()
print("after de-duplication :", len(clean), "rows")

# negative throughput is a counter rollover, not a measurement
neg = clean.throughput_mbps < 0
clean.loc[neg, "throughput_mbps"] = None
print("negative throughput nulled :", int(neg.sum()))
print("blanks left untouched      :", int(clean.latency_ms.isna().sum()))

In [ ]:
clean = raw.drop_duplicates(subset=["cell_id", "reading_ts"], keep="first").copy()
print("after de-duplication :", len(clean), "rows")

# negative throughput is a counter rollover, not a measurement
neg = clean.throughput_mbps < 0
clean.loc[neg, "throughput_mbps"] = None
print("negative throughput nulled :", int(neg.sum()))
print("blanks left untouched      :", int(clean.latency_ms.isna().sum()))


def load(rows):
    cur = con.cursor()
    cur.executemany(
        "INSERT OR IGNORE INTO kpi_readings "
        "(reading_ts, cell_id, region, latency_ms, throughput_mbps, "
        "downtime_min, drop_call_rate_pct) VALUES (?,?,?,?,?,?,?)",
        rows,
    )
    con.commit()


# First load
load(recs)
print(
    "after first load  :",
    con.execute("SELECT COUNT(*) FROM kpi_readings").fetchone()[0],
)

# Second load is intentional: INSERT OR IGNORE should prevent duplicates.
load(recs)
print(
    "after second load :",
    con.execute("SELECT COUNT(*) FROM kpi_readings").fetchone()[0],
)


In [ ]:
import random, time

last = con.execute("SELECT MAX(reading_ts) FROM kpi_readings").fetchone()[0]
cursor_ts = pd.to_datetime(last)
cells = [r[0] for r in con.execute("SELECT DISTINCT cell_id FROM kpi_readings")]
regions = dict(con.execute("SELECT DISTINCT cell_id, region FROM kpi_readings"))
random.seed(9)

for tick in range(4):  # change to: while True
    cursor_ts = cursor_ts + pd.Timedelta(minutes=15)
    batch = [
        (
            cursor_ts.strftime("%Y-%m-%d %H:%M:%S"),
            c,
            regions[c],
            round(random.uniform(26, 74), 2),
            round(random.uniform(90, 300), 2),
            0,
            round(random.uniform(0.2, 1.4), 3),
        )
        for c in cells
    ]
    load(batch)
    total = con.execute("SELECT COUNT(*) FROM kpi_readings").fetchone()[0]
    print(f"tick {tick + 1}: appended {len(batch)} at {cursor_ts} -> {total} rows")
    time.sleep(3)

In [ ]:
# SELECT cell_id,
#        substr(reading_ts, 1, 13) || ':' ||
#        printf('%02d', (CAST(substr(reading_ts, 15, 2) AS INTEGER) / 15) * 15)
#          AS window_start,
#        ROUND(AVG(latency_ms), 2) AS avg_latency,
#        ROUND(MAX(latency_ms), 2) AS peak_latency,
#        COUNT(*)                  AS readings
# FROM kpi_readings
# WHERE cell_id = 'CELL-DEL-03' AND reading_ts LIKE '2026-08-05%'
# GROUP BY cell_id, window_start
# ORDER BY peak_latency DESC
# LIMIT 3;

# SELECT cell_id,
#        substr(reading_ts, 1, 13) || ':00' AS window_start,
#        ROUND(AVG(latency_ms), 2) AS avg_latency,
#        ROUND(MAX(latency_ms), 2) AS peak_latency,
#        COUNT(*)                  AS readings
# FROM kpi_readings
# WHERE cell_id = 'CELL-DEL-03' AND reading_ts LIKE '2026-08-05%'
# GROUP BY cell_id, window_start
# ORDER BY avg_latency DESC
# LIMIT 3;

In [ ]:
peak = con.execute("""SELECT MAX(latency_ms) FROM kpi_readings 
    WHERE cell_id='CELL-DEL-03' AND reading_ts LIKE '2026-08-05%'""").fetchone()[0]

four_h = con.execute(""" 
    SELECT MAX(a) FROM ( 
      SELECT AVG(latency_ms) a FROM kpi_readings 
      WHERE cell_id='CELL-DEL-03' AND reading_ts LIKE '2026-08-05%' 
      GROUP BY CAST(substr(reading_ts,12,2) AS INTEGER)/4)""").fetchone()[0]

n_raw = con.execute("""SELECT COUNT(*) FROM kpi_readings 
    WHERE cell_id='CELL-DEL-03' AND reading_ts LIKE '2026-08-05%' 
    AND latency_ms > 100""").fetchone()[0]

print(f"true 15-minute peak  : {peak:.2f} ms")
print(f"highest 4-hour mean  : {four_h:.2f} ms")
print(f"understated by       : {(peak - four_h) / peak * 100:.1f}%")
print(f"breaches of 100 ms on the raw grain : {n_raw}")
print(f"breaches of 100 ms on the aggregate : {int(four_h > 100)}")

In [ ]:
# Quick database validation
print("total rows :", con.execute("SELECT COUNT(*) FROM kpi_readings").fetchone()[0])
print(
    "duplicate keys :",
    con.execute(
        """
        SELECT COUNT(*)
        FROM (
            SELECT cell_id, reading_ts
            FROM kpi_readings
            GROUP BY cell_id, reading_ts
            HAVING COUNT(*) > 1
        )
        """
    ).fetchone()[0],
)